# Llama 3.2 3B TP=2 sharded-state research artifact

Status: **PREPARED_FOR_KAGGLE**. Use a fresh T4 x2 session, Internet on, accepted model access, and an `HF_TOKEN` Kaggle secret. The Llama artifact upload is blocked pending a separate redistribution review; access is not redistribution permission.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MODEL_KEY='llama32_3b'; WORK=Path('/kaggle/working'); SOURCE=WORK/f'kaggle-vllm-{MODEL_KEY}-source'; RUNTIME=WORK/f'kaggle-vllm-{MODEL_KEY}-runtime'; CACHE=WORK/'kaggle-vllm-cache'; MANIFEST=RUNTIME/'runtime.json'; REF='research/m4-multimodel-tp-crossover-v020'
assert Path('/kaggle').exists() and not SOURCE.exists() and not RUNTIME.exists(), 'Use a fresh Kaggle T4 x2 session'
token=UserSecretsClient().get_secret('HF_TOKEN'); assert token, 'HF_TOKEN secret is required for gated Llama access'; os.environ['HF_TOKEN']=token
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','kaggle-vllm[hub]==0.2.0'],check=True)
subprocess.run(['git','clone','--depth','1','--branch',REF,'https://github.com/kaggle-vllm/kaggle-vllm.git',str(SOURCE)],check=True)
RUNTIME.mkdir(); CACHE.mkdir(exist_ok=True)
boot=['kaggle-vllm','bootstrap','--strict','--staged',str(RUNTIME/'staged'),'--overlay',str(RUNTIME/'overlay'),'--cache',str(CACHE),'--manifest',str(MANIFEST)]
subprocess.run(boot+['--dry-run'],check=True); subprocess.run(boot,check=True)
data=json.loads(MANIFEST.read_text()); RUN_ENV=dict(os.environ); RUN_ENV.update(data['runtime_environment']); RUN_ENV['PYTHONPATH']=str(SOURCE/'src')+os.pathsep+RUN_ENV['PYTHONPATH']; WHEEL=CACHE/data['wheel']['filename']; NOTEBOOK=SOURCE/'kaggle-notebooks/kaggle_vllm_research_llama32_3b_t4x2_sharded.ipynb'
assert WHEEL.is_file() and NOTEBOOK.is_file() and '0.2.0' in (SOURCE/'pyproject.toml').read_text(); print('Secure token present; value was not printed')

In [ ]:
command=[sys.executable,str(SOURCE/'scripts/kaggle_model_artifact.py'),'--model-key',MODEL_KEY,'--working-root',str(WORK),'--native-wheel',str(WHEEL),'--notebook-source',str(NOTEBOOK)]
subprocess.run(command,cwd=SOURCE,env=RUN_ENV,check=True)
evidence=WORK/'llama32-3b-evidence'; assert (evidence/'SHA256SUMS.txt').is_file(); print('Download the evidence directory. Upload remains blocked pending license review:',evidence)